# Build acoustic-unit dictionary

In [ ]:
from __future__ import annotations

import json
import logging
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import yaml

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "configs" / "config.yaml").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root from {start}. "
        "Expected to find configs/config.yaml in the project root."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f) or {}

OUTPUT_DIR = PROJECT_ROOT / CONFIG.get("paths", {}).get("output_dir", "output")
LOG_DIR = OUTPUT_DIR / CONFIG.get("paths", {}).get("log_dir", "output/logs").replace("output/", "")
DICT_DIR = OUTPUT_DIR / CONFIG.get("dictionary", {}).get("output_subdir", "06_dictionary")

DICT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = LOG_DIR / "05_build_unit_dictionary.log"

logger = logging.getLogger("05_build_unit_dictionary")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
logger.addHandler(stream_handler)

logger.info("PROJECT_ROOT=%s", PROJECT_ROOT)
logger.info("CONFIG_PATH=%s", CONFIG_PATH)
logger.info("DICT_DIR=%s", DICT_DIR)


In [ ]:
TOKENS_JSONL = OUTPUT_DIR / CONFIG.get("tokens", {}).get("output_subdir", "05_tokens") / "token_sequences.jsonl"
GLOBAL_FREQ_CSV = OUTPUT_DIR / CONFIG.get("tokens", {}).get("output_subdir", "05_tokens") / "global_token_frequencies.csv"
SEGMENTS_CSV = OUTPUT_DIR / CONFIG.get("segmentation", {}).get("output_subdir", "02_segments") / "segments.csv"
SEGMENT_VECTORS_NPZ = OUTPUT_DIR / CONFIG.get("clustering", {}).get("segment_vectors_subdir", "03_segment_vectors") / "segment_vectors.npz"

assert TOKENS_JSONL.exists(), f"Missing: {TOKENS_JSONL}"
assert GLOBAL_FREQ_CSV.exists(), f"Missing: {GLOBAL_FREQ_CSV}"
assert SEGMENTS_CSV.exists(), f"Missing: {SEGMENTS_CSV}"

print("Inputs found:")
print(" -", TOKENS_JSONL)
print(" -", GLOBAL_FREQ_CSV)
print(" -", SEGMENTS_CSV)
print(" -", SEGMENT_VECTORS_NPZ, "(optional)" if SEGMENT_VECTORS_NPZ.exists() else "(not found; will use fallback prototypes)")


In [ ]:
def safe_str(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s != "" else None

def flatten_token_sequences(jsonl_path: Path) -> pd.DataFrame:
    rows = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            clip_id = record.get("clip_id")
            metadata = record.get("metadata", {}) or {}
            clip_duration_sec = record.get("clip_duration_sec")
            n_segments = record.get("n_segments")
            seq_ids = record.get("sequence_unit_ids", [])
            seq_indices = record.get("sequence_unit_indices", [])
            segments = record.get("segments", [])

            if not (len(seq_ids) == len(seq_indices) == len(segments) == n_segments):
                raise ValueError(f"Length mismatch in token sequence record: {clip_id}")

            for seg in segments:
                row = {
                    "clip_id": clip_id,
                    "clip_duration_sec": clip_duration_sec,
                    "n_segments_in_clip": n_segments,
                    "segment_index": int(seg.get("segment_index")),
                    "segment_id": seg.get("segment_id"),
                    "unit_index": int(seg.get("unit_index")),
                    "unit_id": seg.get("unit_id"),
                    "start_sec": float(seg.get("start_sec")),
                    "end_sec": float(seg.get("end_sec")),
                    "duration_sec": float(seg.get("duration_sec")),
                }
                for key, value in metadata.items():
                    row[key] = value
                rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("No rows found in token_sequences.jsonl")
    return df

def add_global_baselines(df: pd.DataFrame, unit_col: str, value_col: str) -> pd.DataFrame:
    base = df.groupby(value_col).size().rename("global_count").reset_index()
    base["global_proportion"] = base["global_count"] / base["global_count"].sum()

    per_unit = df.groupby([unit_col, value_col]).size().rename("count").reset_index()
    unit_totals = df.groupby(unit_col).size().rename("unit_total").reset_index()

    out = per_unit.merge(unit_totals, on=unit_col, how="left").merge(base, on=value_col, how="left")
    out["unit_proportion"] = out["count"] / out["unit_total"]
    out["lift_vs_global"] = out["unit_proportion"] / out["global_proportion"].replace(0, np.nan)
    return out

def try_load_segment_vectors(npz_path: Path):
    if not npz_path.exists():
        logger.info("segment_vectors.npz not found; prototype selection will fall back to median-duration examples.")
        return None, None

    data = np.load(npz_path, allow_pickle=True)
    keys = set(data.files)

    vector_key_candidates = ["vectors", "segment_vectors", "X", "features", "embeddings"]
    id_key_candidates = ["segment_ids", "ids", "segment_id", "row_ids"]

    vector_key = next((k for k in vector_key_candidates if k in keys), None)
    id_key = next((k for k in id_key_candidates if k in keys), None)

    if vector_key is None or id_key is None:
        logger.info("segment_vectors.npz present but expected keys were not found. Available keys: %s", sorted(keys))
        return None, None

    vectors = data[vector_key]
    segment_ids = data[id_key]

    if len(vectors) != len(segment_ids):
        logger.info("segment_vectors length mismatch: %s vs %s", len(vectors), len(segment_ids))
        return None, None

    logger.info("Loaded segment vectors from %s using keys (%s, %s)", npz_path, vector_key, id_key)
    return vectors, segment_ids

def select_prototypes(unit_df: pd.DataFrame, vectors_lookup: dict | None, top_n: int = 3) -> pd.DataFrame:
    rows = []

    for unit_id, group in unit_df.groupby("unit_id", sort=True):
        group = group.sort_values(["clip_id", "segment_index"]).reset_index(drop=True)

        if vectors_lookup is not None:
            candidate = group[group["segment_id"].isin(vectors_lookup.keys())].copy()
        else:
            candidate = pd.DataFrame()

        if not candidate.empty:
            mat = np.vstack([vectors_lookup[sid] for sid in candidate["segment_id"]])
            centroid = mat.mean(axis=0, keepdims=True)
            distances = np.linalg.norm(mat - centroid, axis=1)
            candidate["prototype_distance"] = distances
            chosen = candidate.nsmallest(min(top_n, len(candidate)), "prototype_distance").copy()
            chosen["prototype_method"] = "nearest_to_unit_centroid"
        else:
            fallback = group.copy()
            median_dur = fallback["duration_sec"].median()
            fallback["prototype_distance"] = (fallback["duration_sec"] - median_dur).abs()
            chosen = fallback.nsmallest(min(top_n, len(fallback)), "prototype_distance").copy()
            chosen["prototype_method"] = "closest_to_unit_median_duration"

        chosen["prototype_rank"] = np.arange(1, len(chosen) + 1)

        keep_cols = [
            "unit_index", "unit_id", "prototype_rank", "prototype_method",
            "segment_id", "clip_id", "segment_index", "start_sec", "end_sec",
            "duration_sec", "main_category_final", "sub_category_final",
            "farm", "barn_zone", "microphone", "mic_placement_context",
            "relative_path", "audio_filename", "file_name"
        ]
        rows.append(chosen[[c for c in keep_cols if c in chosen.columns]])

    if rows:
        return pd.concat(rows, ignore_index=True)
    return pd.DataFrame()

logger.info("Utility functions ready")
print("Helpers loaded.")

In [ ]:
segments_df = pd.read_csv(SEGMENTS_CSV)
global_freq_df = pd.read_csv(GLOBAL_FREQ_CSV)
token_segments_df = flatten_token_sequences(TOKENS_JSONL)

logger.info("Loaded %s rows from token_sequences.jsonl", len(token_segments_df))
logger.info("Loaded %s rows from global_token_frequencies.csv", len(global_freq_df))
logger.info("Loaded %s rows from segments.csv", len(segments_df))

print("token_segments_df shape:", token_segments_df.shape)
print("global_freq_df shape:", global_freq_df.shape)
print("segments_df shape:", segments_df.shape)

display(token_segments_df.head(3))
display(global_freq_df.head(3))

In [ ]:
seg_counts_from_jsonl = token_segments_df.groupby("clip_id").size().rename("count_jsonl")
seg_counts_from_segments = segments_df.groupby("clip_id").size().rename("count_segments")
integrity = seg_counts_from_jsonl.to_frame().join(seg_counts_from_segments, how="outer")

mismatch = integrity[integrity["count_jsonl"] != integrity["count_segments"]]
if not mismatch.empty:
    raise ValueError(f"Mismatch between token_sequences and segments.csv for {len(mismatch)} clips")

freq_check = token_segments_df.groupby("unit_index").size().rename("segment_count_from_jsonl").reset_index()
freq_compare = global_freq_df.merge(freq_check, on="unit_index", how="left")
freq_compare["matches"] = freq_compare["segment_count"] == freq_compare["segment_count_from_jsonl"]

if not freq_compare["matches"].all():
    raise ValueError("global_token_frequencies.csv does not match token_sequences.jsonl")

logger.info("Integrity checks passed")
print("Integrity checks passed.")

In [ ]:
for col in [
    "farm", "barn_zone", "microphone", "mic_placement_context",
    "main_category_final", "sub_category_final"
]:
    if col in token_segments_df.columns:
        token_segments_df[col] = token_segments_df[col].map(safe_str)

token_segments_df["duration_sec"] = token_segments_df["duration_sec"].astype(float)
token_segments_df["unit_index"] = token_segments_df["unit_index"].astype(int)
token_segments_df["segment_index"] = token_segments_df["segment_index"].astype(int)

print("Unique units:", token_segments_df["unit_id"].nunique())
print("Total tokenized segments:", len(token_segments_df))

In [ ]:
unit_inventory = (
    token_segments_df.groupby(["unit_index", "unit_id"], as_index=False)
    .agg(
        segment_count=("segment_id", "count"),
        clip_count=("clip_id", "nunique"),
        mean_duration_sec=("duration_sec", "mean"),
        median_duration_sec=("duration_sec", "median"),
        min_duration_sec=("duration_sec", "min"),
        max_duration_sec=("duration_sec", "max"),
        p10_duration_sec=("duration_sec", lambda s: float(np.percentile(s, 10))),
        p90_duration_sec=("duration_sec", lambda s: float(np.percentile(s, 90))),
        mean_segment_index=("segment_index", "mean"),
    )
)

unit_inventory["segment_proportion"] = unit_inventory["segment_count"] / unit_inventory["segment_count"].sum()
unit_inventory["clip_proportion"] = unit_inventory["clip_count"] / token_segments_df["clip_id"].nunique()

dominant_fields = [
    "main_category_final", "sub_category_final", "farm",
    "barn_zone", "microphone", "mic_placement_context"
]

for field in dominant_fields:
    if field in token_segments_df.columns:
        dom = (
            token_segments_df.groupby(["unit_index", field])
            .size()
            .rename("n")
            .reset_index()
            .sort_values(["unit_index", "n", field], ascending=[True, False, True])
            .drop_duplicates(subset=["unit_index"])
            .rename(columns={field: f"dominant_{field}", "n": f"dominant_{field}_count"})
        )
        unit_inventory = unit_inventory.merge(dom, on="unit_index", how="left")

unit_inventory = unit_inventory.sort_values(["segment_count", "unit_index"], ascending=[False, True]).reset_index(drop=True)

unit_inventory_path = DICT_DIR / "unit_inventory.csv"
unit_inventory.to_csv(unit_inventory_path, index=False)

logger.info("Saved unit_inventory.csv with %s rows", len(unit_inventory))
display(unit_inventory.head(10))
print("Saved:", unit_inventory_path)

In [ ]:
category_frames = []
category_fields = [f for f in ["main_category_final", "sub_category_final"] if f in token_segments_df.columns]

for field in category_fields:
    valid = token_segments_df[token_segments_df[field].notna()].copy()
    if valid.empty:
        continue

    assoc = add_global_baselines(valid, unit_col="unit_id", value_col=field)
    assoc.insert(0, "association_field", field)
    assoc = assoc.rename(columns={field: "association_value"})
    assoc["rank_within_unit_by_count"] = assoc.groupby("unit_id")["count"].rank(method="dense", ascending=False).astype(int)
    assoc["rank_within_unit_by_lift"] = assoc.groupby("unit_id")["lift_vs_global"].rank(method="dense", ascending=False).astype(int)
    category_frames.append(assoc)

unit_category_associations = pd.concat(category_frames, ignore_index=True) if category_frames else pd.DataFrame()
unit_category_associations = unit_category_associations.sort_values(
    ["unit_id", "association_field", "count", "lift_vs_global"],
    ascending=[True, True, False, False]
).reset_index(drop=True)

unit_category_path = DICT_DIR / "unit_category_associations.csv"
unit_category_associations.to_csv(unit_category_path, index=False)

logger.info("Saved unit_category_associations.csv with %s rows", len(unit_category_associations))
display(unit_category_associations.head(12))
print("Saved:", unit_category_path)

In [ ]:
context_frames = []
context_fields = [f for f in ["farm", "barn_zone", "microphone", "mic_placement_context"] if f in token_segments_df.columns]

for field in context_fields:
    valid = token_segments_df[token_segments_df[field].notna()].copy()
    if valid.empty:
        continue

    assoc = add_global_baselines(valid, unit_col="unit_id", value_col=field)
    assoc.insert(0, "context_field", field)
    assoc = assoc.rename(columns={field: "context_value"})
    assoc["rank_within_unit_by_count"] = assoc.groupby("unit_id")["count"].rank(method="dense", ascending=False).astype(int)
    assoc["rank_within_unit_by_lift"] = assoc.groupby("unit_id")["lift_vs_global"].rank(method="dense", ascending=False).astype(int)
    context_frames.append(assoc)

unit_context_associations = pd.concat(context_frames, ignore_index=True) if context_frames else pd.DataFrame()
unit_context_associations = unit_context_associations.sort_values(
    ["unit_id", "context_field", "count", "lift_vs_global"],
    ascending=[True, True, False, False]
).reset_index(drop=True)

unit_context_path = DICT_DIR / "unit_context_associations.csv"
unit_context_associations.to_csv(unit_context_path, index=False)

logger.info("Saved unit_context_associations.csv with %s rows", len(unit_context_associations))
display(unit_context_associations.head(12))
print("Saved:", unit_context_path)

In [ ]:
vectors, segment_ids = try_load_segment_vectors(SEGMENT_VECTORS_NPZ)
vectors_lookup = None

if vectors is not None and segment_ids is not None:
    vectors_lookup = {}
    for sid, vec in zip(segment_ids, vectors):
        vectors_lookup[str(sid)] = np.asarray(vec, dtype=np.float32).reshape(-1)

unit_prototypes = select_prototypes(token_segments_df, vectors_lookup=vectors_lookup, top_n=3)
unit_prototypes_path = DICT_DIR / "unit_prototypes.csv"
unit_prototypes.to_csv(unit_prototypes_path, index=False)

logger.info("Saved unit_prototypes.csv with %s rows", len(unit_prototypes))
display(unit_prototypes.head(12))
print("Saved:", unit_prototypes_path)

In [ ]:
summary = {
    "n_units": int(unit_inventory["unit_id"].nunique()),
    "n_segments": int(unit_inventory["segment_count"].sum()),
    "n_clips": int(token_segments_df["clip_id"].nunique()),
    "most_common_unit": unit_inventory.iloc[0]["unit_id"] if not unit_inventory.empty else None,
    "most_common_unit_segments": int(unit_inventory.iloc[0]["segment_count"]) if not unit_inventory.empty else None,
    "rarest_unit": unit_inventory.iloc[-1]["unit_id"] if not unit_inventory.empty else None,
    "rarest_unit_segments": int(unit_inventory.iloc[-1]["segment_count"]) if not unit_inventory.empty else None,
}
summary_df = pd.DataFrame([summary])
display(summary_df)

logger.info("Summary: %s", summary)
print("Dictionary scaffold build complete.")

In [ ]:
logger.info("Finished 05_build_unit_dictionary notebook")
print("Output files:")
print(" -", unit_inventory_path)
print(" -", unit_category_path)
print(" -", unit_context_path)
print(" -", unit_prototypes_path)
print(" -", LOG_PATH)